# Stage 2B: Unstructured Pruning Comparison

This notebook applies three unstructured pruning methods to Moshi at **30% sparsity**
(matching the structured pruning in `02_structured_pruning.ipynb`).

| ID | Method | Mechanism | Calibration |
|----|--------|-----------|-------------|
| **U1** | Magnitude | Prune smallest \|W\| | None |
| **U2** | Wanda | Prune by \|W\| × \|\|X\|\|₂ | 200 self-play conversations |
| **U3** | SparseGPT | Prune + Hessian weight reconstruction | 200 self-play conversations |

**Memory-efficient approach:** Each method loads a fresh model, prunes it, saves the
checkpoint, then fully deletes the model from RAM before the next method. This avoids
needing 2× model memory (no `deepcopy`).

All variants saved to the **same checkpoint directory** as structured variants.

In [1]:
# === SESSION STARTUP ===
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess, sys, os

# Fetch GitHub token from Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    print("Warning: GITHUB_TOKEN not found in Colab Secrets.")
    GITHUB_TOKEN = ""

REPO_OWNER = "RidwanHR5"
REPO_NAME = "moshilite"

# Construct URL with auth token
if GITHUB_TOKEN:
    REPO = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    REPO = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

REPO_DIR = "/content/moshilite"

# Clone or pull latest code
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

# Install as an editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "-q"], check=True)
print("moshilite package installed successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
moshilite package installed successfully!


In [2]:
import torch, yaml, gc, os, json, time
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

EXPERIMENT_ID = "prune30_v1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REPO_DIR = "/content/moshilite"
WEIGHTS_DIR = "/content/drive/MyDrive/moshilite/upstream_weights/moshiko"

with open(f"{REPO_DIR}/configs/stage2_pruning.yaml") as f:
    config = yaml.safe_load(f)

SPARSITY = config["pruning"]["max_layer_pct"]  # 0.30

# Same checkpoint directory as structured pruning
CHECKPOINT_DIR = f"/content/drive/MyDrive/moshilite/checkpoints/{EXPERIMENT_ID}"
CONVERSATIONS_DIR = "/content/drive/MyDrive/moshilite/self_play_data/conversations"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Sparsity target: {SPARSITY:.0%}")
print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print(f"Device:          {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:             {torch.cuda.get_device_name()}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Sparsity target: 30%
Checkpoint dir:  /content/drive/MyDrive/moshilite/checkpoints/prune30_v1
Device:          cuda
GPU:             NVIDIA L4
VRAM:            23.7 GB


### Helper: Load fresh model + cleanup

In [3]:
from moshilite.analysis.moshi_model import load_moshi_for_analysis
from moshilite.pruning.unstructured_pruning import (
    prune_magnitude, prune_wanda, prune_sparsegpt,
    get_model_sparsity, prepare_calibration_from_self_play,
)


def load_fresh_model():
    """Load a fresh Moshi model in BF16. Call before each pruning method."""
    print("Loading Moshi in BF16...")
    model = load_moshi_for_analysis(
        precision="bf16", device=DEVICE, weights_dir=WEIGHTS_DIR
    )
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Loaded: {n_params/1e9:.3f}B params")
    if DEVICE == "cuda":
        print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
    return model


def cleanup():
    """Force garbage collection and free GPU cache. Call AFTER `del model`."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB")



def save_variant(model, variant_name, method, result, prune_time):
    """Save pruned model to checkpoint dir (same format as structured variants)."""
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"{variant_name}.pt")
    sparsity_stats = get_model_sparsity(model)

    torch.save({
        "model_state_dict": model.state_dict(),
        "variant_name": variant_name,
        "pruning_method": method,
        "pruning_type": "unstructured",
        "sparsity": SPARSITY,
        "actual_sparsity": result.actual_sparsity,
        "param_count": sum(p.numel() for p in model.parameters()),
        "param_billions": round(sum(p.numel() for p in model.parameters()) / 1e9, 3),
        "nonzero_params": sparsity_stats["nonzero_params"],
        "nonzero_param_billions": round(sparsity_stats["nonzero_params"] / 1e9, 3),
        "zeroed_params": result.zeroed_params,
        "pruning_time_s": prune_time,
        "experiment_id": EXPERIMENT_ID,
    }, ckpt_path)

    print(f"Saved: {ckpt_path}")
    print(f"  Total params:    {sum(p.numel() for p in model.parameters())/1e9:.3f}B")
    print(f"  Non-zero params: {sparsity_stats['nonzero_params']/1e9:.3f}B")
    print(f"  Actual sparsity: {result.actual_sparsity:.2%}")
    print(f"  Pruning time:    {prune_time:.1f}s")
    return ckpt_path

print("Helpers ready.")

Helpers ready.


### Load Calibration Data (for Wanda & SparseGPT)

This loads self-play conversations from disk into CPU memory. The data is just
numpy arrays (~a few MB total), so it stays resident across model loads.

In [4]:
from pathlib import Path

conv_path = Path(CONVERSATIONS_DIR)
npz_files = sorted(conv_path.glob("**/*.npz"))
npz_files = [f for f in npz_files if f.name.startswith("conv_")]
print(f"Found {len(npz_files)} self-play conversations")

if len(npz_files) >= 50:
    calibration_data = prepare_calibration_from_self_play(
        conversations_dir=CONVERSATIONS_DIR,
        n_conversations=200,
        seq_len=500,
        batch_size=4,
    )
    print(f"Loaded {len(calibration_data)} calibration batches (CPU memory only)")
else:
    print("WARNING: Not enough self-play conversations for Wanda/SparseGPT.")
    print("  Run 04a_self_play_generation.ipynb first, or")
    print("  only U1 (Magnitude) will be available.")
    calibration_data = None

Found 480 self-play conversations
Loaded 50 calibration batches (CPU memory only)


---
## U1: Magnitude Pruning

Prune the 30% of weights with the smallest absolute value.
No calibration needed. Near-instant.

**Flow:** Load model → prune → save → delete model

In [5]:
print("="*70)
print("  U1: MAGNITUDE PRUNING (30%)")
print("="*70)

# 1. Load fresh model
model = load_fresh_model()

# 2. Prune
t0 = time.time()
result_mag = prune_magnitude(model, sparsity=SPARSITY, per_layer=True)
t_mag = time.time() - t0

# 3. Save
save_variant(model, "magnitude_30pct", "magnitude", result_mag, t_mag)

# 4. Cleanup
del model
cleanup()
print("U2 done.\n")


  U1: MAGNITUDE PRUNING (30%)
Loading Moshi in BF16...
Loaded: 7.688B params
VRAM used: 15.4 GB
Saved: /content/drive/MyDrive/moshilite/checkpoints/prune30_v1/magnitude_30pct.pt
  Total params:    7.688B
  Non-zero params: 4.597B
  Actual sparsity: 30.10%
  Pruning time:    27.3s
VRAM after cleanup: 15.4 GB
U1 done.



---
## U2: Wanda Pruning

Prune by `|W| × ||X||₂` — weights that are small AND whose input features
are rarely activated get pruned first. Single forward pass over calibration data.

**Flow:** Load model → run calibration forward pass → prune → save → delete model

In [5]:
if calibration_data is not None:
    print("="*70)
    print("  U2: WANDA PRUNING (30%)")
    print("="*70)

    # 1. Load fresh model
    model = load_fresh_model()

    # 2. Prune (calibration runs internally)
    t0 = time.time()
    result_wanda = prune_wanda(
        model, calibration_data=calibration_data,
        sparsity=SPARSITY, device=DEVICE,
    )
    t_wanda = time.time() - t0

    # 3. Save
    save_variant(model, "wanda_30pct", "wanda", result_wanda, t_wanda)

    # 4. Cleanup
    del model
    cleanup()
    print("U2 done.\n")
else:
    print("SKIPPED U2 (Wanda) - no calibration data.")

  U2: WANDA PRUNING (30%)
Loading Moshi in BF16...
Loaded: 7.688B params
VRAM used: 15.4 GB


Wanda calibration: 100%|██████████| 50/50 [00:56<00:00,  1.12s/it]


Saved: /content/drive/MyDrive/moshilite/checkpoints/prune30_v1/wanda_30pct.pt
  Total params:    7.688B
  Non-zero params: 4.604B
  Actual sparsity: 30.00%
  Pruning time:    106.2s
VRAM after cleanup: 0.0 GB
U2 done.



---
## U3: SparseGPT Pruning

The strongest method. Prunes weights then **reconstructs remaining weights** using
Hessian information to minimize output distortion per layer. Most compute-intensive
(~1-2 hours on L4).

**Flow:** Load model → run calibration → Hessian per layer → prune+reconstruct → save → delete

In [5]:
if calibration_data is not None:
    print("="*70)
    print("  U3: SPARSEGPT PRUNING (30%)")
    print("="*70)
    print("  This is the slowest method (~1-2 hours). Be patient.")

    # 1. Load fresh model
    model = load_fresh_model()

    # 2. Prune (calibration + Hessian computed internally)
    t0 = time.time()
    result_sgpt = prune_sparsegpt(
        model, calibration_data=calibration_data,
        sparsity=SPARSITY, block_size=128,
        percdamp=0.01, device=DEVICE,
    )
    t_sgpt = time.time() - t0

    # 3. Save
    save_variant(model, "sparsegpt_30pct", "sparsegpt", result_sgpt, t_sgpt)

    # 4. Cleanup
    del model
    cleanup()
    print("U3 done.\n")
else:
    print("SKIPPED U3 (SparseGPT) - no calibration data.")

  U3: SPARSEGPT PRUNING (30%)
  This is the slowest method (~1-2 hours). Be patient.
Loading Moshi in BF16...
Loaded: 7.688B params
VRAM used: 15.4 GB


SparseGPT pruning: 100%|██████████| 128/128 [03:09<00:00,  1.48s/it]


Saved: /content/drive/MyDrive/moshilite/checkpoints/prune30_v1/sparsegpt_30pct.pt
  Total params:    7.688B
  Non-zero params: 4.686B
  Actual sparsity: 28.74%
  Pruning time:    1146.5s
VRAM after cleanup: 0.0 GB
U3 done.



---
## Verification

Load each saved checkpoint (metadata only, NOT the full model) and verify
the sparsity was applied correctly.

In [6]:
print("="*70)
print("  VERIFICATION")
print("="*70)

unstructured_variants = [
    ("magnitude_30pct", "Magnitude"),
    ("wanda_30pct", "Wanda"),
    ("sparsegpt_30pct", "SparseGPT"),
]

for variant_name, method_name in unstructured_variants:
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"{variant_name}.pt")
    if not os.path.exists(ckpt_path):
        print(f"  {method_name:12s}: SKIPPED (no checkpoint)")
        continue

    # Load checkpoint metadata (not the state_dict weights)
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    # Count zeros in the state dict to verify sparsity
    sd = ckpt["model_state_dict"]
    total_w, zero_w = 0, 0
    for key, tensor in sd.items():
        if "transformer" in key and "weight" in key and tensor.dim() == 2:
            total_w += tensor.numel()
            zero_w += (tensor == 0).sum().item()

    verified_sparsity = zero_w / total_w if total_w > 0 else 0
    saved_sparsity = ckpt.get("actual_sparsity", -1)
    match = abs(verified_sparsity - saved_sparsity) < 0.01

    status = "PASS" if match else "MISMATCH"
    print(f"  {method_name:12s}: sparsity={verified_sparsity:.2%} "
          f"(saved={saved_sparsity:.2%}) [{status}]  "
          f"size={os.path.getsize(ckpt_path)/1e9:.2f} GB")

    del ckpt, sd
    gc.collect()

  VERIFICATION
  Magnitude   : sparsity=30.10% (saved=30.10%) [PASS]  size=15.38 GB
  Wanda       : sparsity=30.00% (saved=30.00%) [PASS]  size=15.38 GB
  SparseGPT   : sparsity=28.74% (saved=28.74%) [PASS]  size=15.38 GB


---
## Summary: All Variants in Checkpoint Directory

In [8]:
import os, gc

print("\n" + "="*90)
print("  ALL PRUNED VARIANTS")
print("="*90)

# Quick load: only unstructured variants (need torch.load for metadata)
unstructured = ["magnitude_30pct.pt", "wanda_30pct.pt", "sparsegpt_30pct.pt"]

print(f"\n{'Variant':<25} {'Type':<15} {'Method':<12} {'Params (B)':<12} {'Non-Zero (B)':<14} {'Sparsity':<10} {'Time (s)'}")
print("-"*100)

for f in sorted(os.listdir(CHECKPOINT_DIR)):
    if not f.endswith(".pt"):
        continue
    ckpt_path = os.path.join(CHECKPOINT_DIR, f)
    size_gb = os.path.getsize(ckpt_path) / 1e9

    if f in unstructured:
        # Only fully load unstructured (we need the metadata)
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        print(f"{ckpt.get('variant_name','?'):<25} {'Unstructured':<15} "
              f"{ckpt.get('pruning_method','?'):<12} "
              f"{ckpt.get('param_billions','-'):<12} "
              f"{ckpt.get('nonzero_param_billions','-'):<14} "
              f"{ckpt.get('actual_sparsity',0):.1%}{'':>4} "
              f"{ckpt.get('pruning_time_s',0):.0f}")
        del ckpt
        gc.collect()
    else:
        # Structured: just show filename and size (skip slow torch.load)
        name = f.replace(".pt", "")
        print(f"{name:<25} {'Structured':<15} {'—':<12} {'—':<12} {'—':<14} {'0%':<10} —")

print(f"\nCheckpoint dir: {CHECKPOINT_DIR}")
print("Unstructured pruning complete!")



  ALL PRUNED VARIANTS

Variant                   Type            Method       Params (B)   Non-Zero (B)   Sparsity   Time (s)
----------------------------------------------------------------------------------------------------
magnitude_30pct           Unstructured    magnitude    7.688        4.597          30.1%     27
sparsegpt_30pct           Unstructured    sparsegpt    7.688        4.686          28.7%     1147
v1_scattered_nonuni       Structured      —            —            —              0%         —
v1_scattered_uni          Structured      —            —            —              0%         —
v2a_cont_strict_nonuni    Structured      —            —            —              0%         —
v2a_cont_strict_uni       Structured      —            —            —              0%         —
v2b_cont_penalized_nonuni Structured      —            —            —              0%         —
v2b_cont_penalized_uni    Structured      —            —            —              0%         —
v2